# BERT Models Embeddings Generation
This notebook encodes sentences from a .csv file using multiple BERT embedding models:
- BERT (base)
- RoBERTa (base)
- NeoBERT
- ModernBERT
- AraBERT (base)
- AraBERT (large)
- OpenAI Ada
- OpenAI text-embedding-3 (Large)
- EmbeddingGemma-300m
- Voyage 3 (Large)
- BAAI BGE-M3
- e5-large-v2

Embeddings are saved to separate CSV files for each model.

## Setup and Imports


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
from tqdm.auto import tqdm
import warnings
import os
warnings.filterwarnings('ignore')

# Check for CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Install additional dependencies if needed
# Uncomment if you need to install:
# !pip install openai voyageai sentence-transformers

Using device: cpu


## Load Data


In [3]:
# Load the sentences dataset
df = pd.read_csv('/Users/sattam/Downloads/AraVAD/Datasets/extreme_pole_sentences_arabic.csv')
print(f"Loaded {len(df)} sentences")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()


Loaded 90 sentences

Columns: ['sentence', 'pos/neg', 'V/A/D']

First few rows:


,sentence,pos/neg,V/A/D
0,عناق حبيبي الطري يملأني بسعادة هادئة نقية أجسا...,pos,V
1,الجلوس في الكنيسة يغسلني الوعظ السلمي مسلماً ل...,pos,V
2,كلمات أصدقائي اللطيفة ترفع من روحي بلطف تجعلني...,pos,V
3,مشاهدة غروب الشمس مع عائلتي تملأني موجة من الس...,pos,V
4,طعم وجبتي المفضلة يجلب لي متعة هادئة بسيطة أتذ...,pos,V


## Helper Function for Encoding


In [4]:
def encode_sentences(model_name, sentences, device, batch_size=8):
    """
    Encode sentences using a specified model.

    Args:
        model_name: HuggingFace model identifier
        sentences: List of sentences to encode
        device: torch device (cpu/cuda)
        batch_size: Number of sentences to process at once

    Returns:
        numpy array of embeddings (shape: n_sentences x embedding_dim)
    """
    print(f"\nLoading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(sentences), batch_size), desc=f"Encoding with {model_name}"):
            batch = sentences[i:i+batch_size]

            # Tokenize
            inputs = tokenizer(batch, padding=True, truncation=True,
                             max_length=512, return_tensors='pt').to(device)

            # Get embeddings (use [CLS] token)
            outputs = model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)

    # Clear memory
    del model, tokenizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return np.vstack(embeddings)


## Helper Function to Save Embeddings


## Helper Function for API-based Models


In [ ]:
def encode_sentences_openai(sentences, model_name, api_key=None, batch_size=100):
    """
    Encode sentences using OpenAI API.
    
    Args:
        sentences: List of sentences to encode
        model_name: OpenAI model name (e.g., 'text-embedding-ada-002', 'text-embedding-3-large')
        api_key: OpenAI API key (if None, will use OPENAI_API_KEY env variable)
        batch_size: Number of sentences to process at once
    
    Returns:
        numpy array of embeddings
    """
    try:
        from openai import OpenAI
    except ImportError:
        print("OpenAI library not installed. Install with: pip install openai")
        return None
    
    if api_key is None:
        api_key = os.getenv('OPENAI_API_KEY')
    
    if not api_key:
        print("Warning: OPENAI_API_KEY not set. Please set it as an environment variable.")
        return None
    
    client = OpenAI(api_key=api_key)
    embeddings = []
    
    for i in tqdm(range(0, len(sentences), batch_size), desc=f"Encoding with {model_name}"):
        batch = sentences[i:i+batch_size]
        response = client.embeddings.create(
            input=batch,
            model=model_name
        )
        batch_embeddings = [data.embedding for data in response.data]
        embeddings.extend(batch_embeddings)
    
    return np.array(embeddings)


def encode_sentences_voyage(sentences, model_name, api_key=None, batch_size=128):
    """
    Encode sentences using Voyage AI API.
    
    Args:
        sentences: List of sentences to encode
        model_name: Voyage model name (e.g., 'voyage-3-large')
        api_key: Voyage API key (if None, will use VOYAGE_API_KEY env variable)
        batch_size: Number of sentences to process at once
    
    Returns:
        numpy array of embeddings
    """
    try:
        import voyageai
    except ImportError:
        print("Voyage AI library not installed. Install with: pip install voyageai")
        return None
    
    if api_key is None:
        api_key = os.getenv('VOYAGE_API_KEY')
    
    if not api_key:
        print("Warning: VOYAGE_API_KEY not set. Please set it as an environment variable.")
        return None
    
    vo = voyageai.Client(api_key=api_key)
    embeddings = []
    
    for i in tqdm(range(0, len(sentences), batch_size), desc=f"Encoding with {model_name}"):
        batch = sentences[i:i+batch_size]
        result = vo.embed(batch, model=model_name)
        embeddings.extend(result.embeddings)
    
    return np.array(embeddings)


In [5]:
def save_embeddings_to_csv(df, embeddings, model_name, output_path):
    """
    Save embeddings along with original columns to CSV.

    Args:
        df: Original dataframe with sentence, pos/neg, V/A/D columns
        embeddings: numpy array of embeddings
        model_name: Name of the model (for column naming)
        output_path: Path to save the CSV
    """
    # Create a new dataframe with original columns
    result_df = df.copy()

    # Add embedding dimensions as columns
    embedding_dim = embeddings.shape[1]
    for i in range(embedding_dim):
        result_df[f'emb_{i}'] = embeddings[:, i]

    # Save to CSV
    result_df.to_csv(output_path, index=False)
    print(f"Saved {len(result_df)} rows with {embedding_dim}-dimensional embeddings to {output_path}")


## 1. BERT (base-uncased)


In [7]:
# BERT
bert_embeddings = encode_sentences(
    model_name='bert-base-uncased',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=bert_embeddings,
    model_name='bert',
    output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_bert.csv'
)


Loading bert-base-uncased...


Encoding with bert-base-uncased:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 90 rows with 768-dimensional embeddings to /Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_bert.csv


## 2. RoBERTa (base)


In [8]:
# RoBERTa
roberta_embeddings = encode_sentences(
    model_name='roberta-base',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=roberta_embeddings,
    model_name='roberta',
    output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_roberta.csv'
)


Loading roberta-base...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Encoding with roberta-base:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 90 rows with 768-dimensional embeddings to /Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_roberta.csv


## 3. NeoBERT


In [9]:
%pip install xformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 15.3 MB/s eta 0:00:0000:010:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached torch-2.9.1-cp312-none-macosx_11_0_arm64.whl.metadata (30 kB)
Using cached torch-2.9.1-cp312-none-macosx_11_0_arm64.whl (74.5 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for xformers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [247 lines of output]
      /private/var/folders/pj/88mbw0h90rv6cb568zh6r3nr0000gn/T/pip-build-env-n_s57fhw/overlay/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
        cpu = _conversion_method_template(device=torch.device("cpu"))
      /private/var/folders/pj/88mbw0h90rv6cb568zh6r3nr0000gn

In [10]:
# NeoBERT (using the official model from HuggingFace)
neobert_embeddings = encode_sentences(
    model_name='chandar-lab/NeoBERT',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=neobert_embeddings,
    model_name='neobert',
    output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_neobert.csv'
)


Loading chandar-lab/NeoBERT...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/928 [00:00<?, ?B/s]

model.py: 0.00B [00:00, ?B/s]

Encountered exception while importing xformers: No module named 'xformers'


ImportError: This modeling file requires the following packages that were not found in your environment: xformers. Run `pip install xformers`

## 4. ModernBERT


In [11]:
# ModernBERT (using the base model)
modernbert_embeddings = encode_sentences(
    model_name='answerdotai/ModernBERT-base',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=modernbert_embeddings,
    model_name='modernbert',
    output_path='/content/extreme_pole_embeddings_modernbert.csv'
)



Loading answerdotai/ModernBERT-base...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Encoding with answerdotai/ModernBERT-base:   0%|          | 0/12 [00:00<?, ?it/s]

W1017 09:52:34.010000 278 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Saved 90 rows with 768-dimensional embeddings to /content/extreme_pole_embeddings_modernbert.csv


## 5. AraBERT (base)


In [ ]:
# AraBERT base
arabert_base_embeddings = encode_sentences(
    model_name='aubmindlab/bert-base-arabertv2',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=arabert_base_embeddings,
    model_name='arabert_base',
    output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_arabert_base.csv'
)


## 6. AraBERT (large)


In [ ]:
# AraBERT large
arabert_large_embeddings = encode_sentences(
    model_name='aubmindlab/bert-large-arabertv2',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=arabert_large_embeddings,
    model_name='arabert_large',
    output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_arabert_large.csv'
)


## 7. OpenAI Ada


In [ ]:
# OpenAI Ada (text-embedding-ada-002)
# Note: Requires OPENAI_API_KEY environment variable to be set
openai_ada_embeddings = encode_sentences_openai(
    sentences=df['sentence'].tolist(),
    model_name='text-embedding-ada-002'
)

if openai_ada_embeddings is not None:
    save_embeddings_to_csv(
        df=df,
        embeddings=openai_ada_embeddings,
        model_name='openai_ada',
        output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_openai_ada.csv'
    )
else:
    print("Skipping OpenAI Ada - API key not available or library not installed")


## 8. OpenAI text-embedding-3-large


In [ ]:
# OpenAI text-embedding-3-large
# Note: Requires OPENAI_API_KEY environment variable to be set
openai_3_large_embeddings = encode_sentences_openai(
    sentences=df['sentence'].tolist(),
    model_name='text-embedding-3-large'
)

if openai_3_large_embeddings is not None:
    save_embeddings_to_csv(
        df=df,
        embeddings=openai_3_large_embeddings,
        model_name='openai_3_large',
        output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_openai_3_large.csv'
    )
else:
    print("Skipping OpenAI text-embedding-3-large - API key not available or library not installed")


## 9. Google EmbeddingGemma-300m


In [ ]:
# EmbeddingGemma-300m
# Note: This model may require accepting terms on HuggingFace Hub
gemma_embeddings = encode_sentences(
    model_name='google/gemma-2-300m',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=gemma_embeddings,
    model_name='gemma_300m',
    output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_gemma_300m.csv'
)


## 10. Voyage 3 (Large)


In [ ]:
# Voyage 3 Large
# Note: Requires VOYAGE_API_KEY environment variable to be set
voyage_embeddings = encode_sentences_voyage(
    sentences=df['sentence'].tolist(),
    model_name='voyage-3-large'
)

if voyage_embeddings is not None:
    save_embeddings_to_csv(
        df=df,
        embeddings=voyage_embeddings,
        model_name='voyage_3_large',
        output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_voyage_3_large.csv'
    )
else:
    print("Skipping Voyage 3 Large - API key not available or library not installed")


## 11. BAAI BGE-M3


In [ ]:
# BAAI BGE-M3 (Multilingual model)
bge_m3_embeddings = encode_sentences(
    model_name='BAAI/bge-m3',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=bge_m3_embeddings,
    model_name='bge_m3',
    output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_bge_m3.csv'
)


## 12. e5-large-v2


In [ ]:
# e5-large-v2 (Text Embeddings by Text Representations)
e5_large_embeddings = encode_sentences(
    model_name='intfloat/e5-large-v2',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=e5_large_embeddings,
    model_name='e5_large_v2',
    output_path='/Users/sattam/Downloads/AraVAD/Datasets/Embeddings/ar_extreme_pole_embeddings_e5_large_v2.csv'
)


## Summary: Embedding Dimensions


In [ ]:
print("=" * 60)
print("EMBEDDING DIMENSIONS SUMMARY")
print("=" * 60)

# HuggingFace models
print("\n--- Transformer Models (HuggingFace) ---")
print(f"  1. BERT (base):              {bert_embeddings.shape[1]} dims")
print(f"  2. RoBERTa (base):           {roberta_embeddings.shape[1]} dims")
print(f"  3. NeoBERT:                  {neobert_embeddings.shape[1]} dims")
print(f"  4. ModernBERT:               {modernbert_embeddings.shape[1]} dims")
print(f"  5. AraBERT (base):           {arabert_base_embeddings.shape[1]} dims")
print(f"  6. AraBERT (large):          {arabert_large_embeddings.shape[1]} dims")
print(f"  9. EmbeddingGemma-300m:      {gemma_embeddings.shape[1]} dims")
print(f" 11. BAAI BGE-M3:              {bge_m3_embeddings.shape[1]} dims")
print(f" 12. e5-large-v2:              {e5_large_embeddings.shape[1]} dims")

# API-based models
print("\n--- API-based Models ---")
if openai_ada_embeddings is not None:
    print(f"  7. OpenAI Ada:               {openai_ada_embeddings.shape[1]} dims")
else:
    print(f"  7. OpenAI Ada:               [Not available]")

if openai_3_large_embeddings is not None:
    print(f"  8. OpenAI 3-large:           {openai_3_large_embeddings.shape[1]} dims")
else:
    print(f"  8. OpenAI 3-large:           [Not available]")

if voyage_embeddings is not None:
    print(f" 10. Voyage 3 (Large):         {voyage_embeddings.shape[1]} dims")
else:
    print(f" 10. Voyage 3 (Large):         [Not available]")

print("=" * 60)


## Embedding Dimensions


In [12]:
print(f"Embedding dimensions:")
print(f"  - BERT: {bert_embeddings.shape[1]}")
print(f"  - RoBERTa: {roberta_embeddings.shape[1]}")
print(f"  - NeoBERT: {neobert_embeddings.shape[1]}")
print(f"  - ModernBERT: {modernbert_embeddings.shape[1]}")


Embedding dimensions:
  - BERT: 768
  - RoBERTa: 768
  - NeoBERT: 768
  - ModernBERT: 768
